## Datasets and Dataloaders

Dataset and Dataloader are basic classes for data loading in pytorch. Dataset stores the samples and their corresponding labels, and DataLoader wraps an iterable around the Dataset to enable easy access to the samples.

**Datasets**

A typical Dataset for supervised ML returns two tensors: x and y, where y must be predicted based on x.

A dataset can be really simple. It only has to implement two methods: \_\_len\_\_() and \_\_get_item\_\_()

**Task A**
Create a pytorch Dataset that returns a pair $x, y$ such that 

$$
  x = [x_1, x_2] \newline
$$

$$
  x_1 \sim \it{U}[-1, 1] \newline
$$
$$
  x_2 \sim \it{U}[-1, 1] \newline
$$

$$y = \begin{cases} 0 & \text{if } x_1 < -0.3 \\ 1 & \text{if } x_1 \ge 0.3 \text{, } x_2 < 0 \\ 2 & \text{if } x_1 \ge 0.3 \text{, } x_2 \ge 0 \end{cases}$$

When initiated with SimpleDataset(100), it should generate 100 samples, which then can then be accessed with \_\_get_item\_\_(idx).

On this lab, feel free to use chatbots for help.

In [1]:
import torch
from torch.utils.data import Dataset
from typing import Tuple, Any

class SimpleDataset(Dataset):
    def __init__(self, num_samples: int):
        self.num_samples = num_samples
        self.x = torch.rand(num_samples, 2) * 2 - 1
        x1 = self.x[:, 0]
        x2 = self.x[:, 1]
        self.y = torch.where(x1 < -0.3, 0, torch.where(x1 >= 0.3, torch.where(x2 < 0, 1, 2), 1))

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]

In [6]:

from simple_visualizations import visualize_dataset

dataset = SimpleDataset(100)
visualize_dataset(dataset)

alt.Chart(...)

**DataLoaders**

DataLoaders wrap an iterable over a dataset to provide an efficient and easy way to access the data.


Their primary responsibilities include:

- Batching the data: grouping individual data samples into batches of a specified size. This is necessary for efficient model training

- Shuffling the data: shuffling the data at every pass over the entire dataset

- Parallel Data Loading: using multiprocessing to load data in parallel (with the num_workers argument), speeding up the data fetching process

- Automatic Memory Pinning: moving fetched data to CUDA pinned memory which makes training faster

Initialization of a default DataLoader with batch_size=16:

In [7]:
from torch.utils.data import DataLoader

dataset = SimpleDataset(
    num_samples=100
)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

**Task B** Try to iterate through the dataloader. How does batch size affect the shape of returned items? When does the iteration end? 

In [9]:
for x, y in dataloader:
    print(x.shape, y.shape)


torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([16, 2]) torch.Size([16])
torch.Size([4, 2]) torch.Size([4])


The last batch has an awkward shape. It can be skipped by setting drop_last=True when instantiating the dataloader.

## A linear model

**nn.Module**

All architectural components in pytorch inherit from nn.Module, from the simplest nn.Relu to the most complex nn.Transformer. Modules can also be a part of other modules. See https://pytorch.org/docs/stable/nn.html for the list of all modules.

The simplest module implements only one method: `forward`, which takes in the input, does whatever it's supposed to do, and returns the result.

**Task C** Implement a custom module with the ReLU activation function.


In [11]:
class ReLU(torch.nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.clamp(x, min=0)


In [ ]:
from simple_visualizations import visualize_module

relu = ReLU()

X = torch.tensor(1)
result = relu(X)  # invokes .forward(X)
assert result.ndim == 0
assert result.item() == 1

X = torch.tensor([-1, 2])
result = relu(X)
assert result.ndim == 1
assert result[0].item() == 0
assert result[1].item() == 2

X = torch.arange(-50, 50, 0.1)
visualize_module(X, relu)


**Softmax**

Our dataset is made for classification. Because we're using pytorch, our classifier will be expected to return the predicted *logits* of each class. If we then wanted to display the predicted *probabilities* (summing up to 1), we'd have to apply the *Softmax* function on the logits.

**Task D** Implement Softmax

$$
\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{K} e^{x_j}}
$$

Perform this operation along the last dimension, treating all others as batch dimensions. 

In [ ]:
class Softmax(torch.nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError('Implement softmax across the last dimension')


In [ ]:
softmax = Softmax()

X = torch.tensor([0, 0])
result = softmax(X)
assert result.ndim == 1
assert result[0].item() == 0.5
assert result[1].item() == 0.5

X = torch.tensor([-1, 1, 2])
result = softmax(X)
assert abs(result[0].item() - 0.0351) < 0.001
assert abs(result[1].item() - 0.2595) < 0.001
assert abs(result[2].item() - 0.7054) < 0.001


X = torch.tensor([[0, 0, 0], [-1, 1, 2]])
result = softmax(X)
assert abs(result[0][0].item() - 0.3333) < 0.001
assert abs(result[0][1].item() - 0.3333) < 0.001
assert abs(result[0][2].item() - 0.3333) < 0.001
assert abs(result[1][0].item() - 0.0351) < 0.001
assert abs(result[1][1].item() - 0.2595) < 0.001
assert abs(result[1][2].item() - 0.7054) < 0.001


**nn.Parameter**

Not all modules are stateless like ReLU and Softmax. Some, like a linear layer, contain trainable parameters that, once initizalized, should be tracked and updated.

**nn.Parameter** is an extension of Tensor with some special properties for doing this, mostly connected to training:

- Auto-registration: if a Parameter is an attribute of a Module, it will be auto-detected by torch and returned when module.parameters() is called.
- Gradient tracking by default: the requires_grad attribute, used to store gradients in training, is set on by default.
- Optimizer integration: optimizers detect all nn.Parameters by default and update them during training.
- State dict inclusion: saving the model to a file includes all its nn.Parameters.

Typically, all parameters are initialized in the `__init__` method of the module and then used and updated throughout its lifetime.

**Task E** Create a linear/fully-connected layer that calculates the function $y=xW^T+b$, where $W$ and $b$ are trainable parameters. Initialize the weights and the bias from $\it{U}[-\frac{1}{\sqrt{N_{in}}}, \frac{1}{\sqrt{N_{in}}}]$.

(this is called the He initialization, we'll explain the reasons behind it on Lecture 3)

In [ ]:
class Linear(torch.nn.Module):
    def __init__(self, in_features: int, out_features: int):
        # make sure to call the parameters self.bias and self.weight
        super().__init__()
        raise NotImplementedError('Initialize parameters')

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: shape (*, in_features)
        Return: shape (*, out_features)
        """
        raise NotImplementedError('Implement forward')


In [ ]:
# compare to the official pytorch torch.nn.Linear

x = torch.rand((10, 16))
ours = Linear(16, 8)
assert ours.weight.ndim == 2
assert ours.bias.ndim == 1

official = torch.nn.Linear(16, 8, bias=True)
# here, we're copying the weight and bias from one model to the other
# using the fact that they are all registered as parameters
ours.load_state_dict(official.state_dict())

actual_output = ours(x)
expected_output = official(x)
error = torch.abs(actual_output - expected_output).sum().item()
assert error < 0.001


Let's create our first model: a single linear layer.

In [ ]:
model = Linear(
    in_features=2,  # two features in our dataset
    out_features=3  # three classes
)

output = model(torch.tensor([0.5, 0.5]))
output_probabilities = Softmax()(output)
print(
    f'The model outputs {output} for point {0.5, 0.5}, '
    f'which means probabilities {output_probabilities}'
)

It's not trained yet, so the returned values are random. Let's train it on our dataset.

In [ ]:
from simple_model_train import train_model  # let's hide the training code for today

train_model(model, dataloader, num_training_iterations=10, is_classification=True)
# 10 is the number of times the dataloaded will be traversed.
# A single pass through a dataset is called an epoch.


In this directory, you'll find a small dashboard that visualizes training history, made with a library called streamlit. Run '**streamlit run network_dashboard.py**' in a terminal to see it.

Our classifier is just a linear function. This would normally make it a *linear classifier*. However, because of how pytorch expect classifiers to return logits and actually applies Softmax on the output, our model is a *logistic classifier* – a linear classifier followed by a Softmax. The fact that the model returns logits, not probabilities, seems like a weird pytorch quirk, but it's there because of a better numerical stability. The Softmax addition is only when applying the loss function inside the training code; if we want to get probabilities from our model, we can use our Softmax function.

**Task F** See what probabilities the model returns for the point $[0.5, 0.5]$ now.

In [ ]:
# your code here

## A non-linear model

Linear models aren't very powerful. Now, we'll see a classic problem they can't solve.

**Task G**
Create a pytorch Dataset that returns a pair $x, y$.

$$
  x = [x_1, x_2] \newline
$$

$$
  x_1 \sim \it{U}[-1, 1] \newline
$$
$$
  x_2 \sim \it{U}[-1, 1] \newline
$$

$$y = \begin{cases} 0 & \text{if sign(} x_1 \text{) } = \text{sign(} x_2 \text{)} \\ 1 & \text{if sign(} x_1 \text{) } \neq \text{sign(} x_2 \text{)} \end{cases}$$

Instead of generating a finite set on initialization, this dataset should generate a **completely new sample** every time \_\_get_item\_\_(idx) is called. This means that
1. only a single sample must be generated every iteration
2. idx in \_\_get_item\_\_ should be ignored
3. the length serves only as an info for the dataloader

This aims to show that not all datasets load the data; some can generate it on the fly.

Feel free to use a chatbot for this one if you're running low on time.

In [ ]:

class XORDataset(Dataset):
    def __init__(self, num_samples: int):
        self.num_samples = num_samples

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        raise NotImplementedError("Place for code")


In [ ]:
dataset = XORDataset(128)
visualize_dataset(dataset)

**Task H** Create a logistic classifier and try to train it on this dataset (don't expect it to work too well).

In [ ]:
# code here

Our model fails because a linear function can't approximate XOR well. To solve this, we'll add a nonlinear function, followed by a new linear layer (which will be called a *hidden layer*, because it's not a part of the input or the output).

To concatenate multiple modules, we'll use torch.nn.Sequential. This module, created by passing a list of modules to its constructor, runs them one after another, passing the results of the *i-th* one to the *i+1-th*.

It was fun implementing our own modules, but feel free to use torch.nn.Linear for the linear layer and torch.nn.ReLU for the activation instead.

**Task I** Create a nn.Sequential model with a linear layer, nn.ReLU and another linear layer. Train it with dataset size 256, batch size 32 and lr=0.03.

In [ ]:
# code here